# Crop Condition Signal Research
### Quantitative Research - Agriculture Commodity Signals

**Objective:** Generate tradable signals from USDA weekly Crop Condition & Progress reports.

In [1]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import numpy as np
from datetime import datetime
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

from my_holiday.cbt import CBT
from my_holiday.holiday_utils import apply_date_rules
from data.data_loader import PriceDataLoader
from data.data_loader3 import CropCnPDataLoader
from pages.other_reports.crop_condition_and_progress.signal_crop_cnp import SignalCropCnp

print('All imports successful')

All imports successful


---
## 1. Data Loading
Load crop condition signals, raw condition/progress data, and futures prices.

In [2]:
COMMODITIES = {
    'CORN':    {'price_cmd': 'C', 'contract_month': 'Z', 'name': 'Corn'},
    'SOYBEANS': {'price_cmd': 'S', 'contract_month': 'X', 'name': 'Soybeans'}
}
MONTH_CODE = {1:'F',2:'G',3:'H',4:'J',5:'K',6:'M',7:'N',8:'Q',9:'U',10:'V',11:'X',12:'Z'}
FORWARD_HORIZONS = [5, 10, 20]
SEASON_PHASES = {
    'planting': [4, 5], 'growing': [6, 7, 8],
    'harvest': [9, 10, 11], 'off_season': [12, 1, 2, 3]
}

signal_calc = SignalCropCnp(signal_config=None)
all_signals = {}
for cmd in COMMODITIES:
    print(f'Loading {cmd} signals...')
    df = signal_calc.calculate_signal_all(cmd=cmd)
    df.dropna(subset=['signal_value'], inplace=True)
    all_signals[cmd] = df
    print(f'  -> {len(df)} rows, signals: {df["signal_name"].unique().tolist()}')

Loading CORN signals...
{"asctime": "2026-04-09 17:22:42,361", "levelname": "INFO", "name": "app", "message": "Data loaded from cache for page: crop_condition_and_progress", "pathname": "c:\\Users\\yuhang.hou\\projects\\multipage_dash\\olam_dash\\data\\data_loader3.py", "lineno": 129, "process": 16084, "threadName": "MainThread"}
  -> 2208 rows, signals: ['crop_condition_normalized', 'crop_condition_normalized_diff', 'crop_condition_historical_surprise']
Loading SOYBEANS signals...
{"asctime": "2026-04-09 17:22:44,726", "levelname": "INFO", "name": "app", "message": "Data loaded from cache for page: crop_condition_and_progress", "pathname": "c:\\Users\\yuhang.hou\\projects\\multipage_dash\\olam_dash\\data\\data_loader3.py", "lineno": 129, "process": 16084, "threadName": "MainThread"}
  -> 1960 rows, signals: ['crop_condition_normalized', 'crop_condition_normalized_diff', 'crop_condition_historical_surprise']


In [3]:
raw_loader = CropCnPDataLoader()
raw_condition, raw_progress = {}, {}

for cmd in COMMODITIES:
    print(f'\n--- {cmd} ---')
    raw_condition[cmd] = raw_loader.load_data(cmd, 'condition')
    print(f'  Condition: {len(raw_condition[cmd])} rows')
    try:
        raw_progress[cmd] = raw_loader.load_data(cmd, 'progress')
        print(f'  Progress: {len(raw_progress[cmd])} rows, types: {raw_progress[cmd]["type"].unique()}')
    except Exception as e:
        raw_progress[cmd] = None
        print(f'  Progress: Not available')


--- CORN ---
{"asctime": "2026-04-09 17:22:47,046", "levelname": "INFO", "name": "app", "message": "Data loaded from cache for page: crop_condition_and_progress", "pathname": "c:\\Users\\yuhang.hou\\projects\\multipage_dash\\olam_dash\\data\\data_loader3.py", "lineno": 129, "process": 16084, "threadName": "MainThread"}
  Condition: 106548 rows
{"asctime": "2026-04-09 17:22:47,096", "levelname": "INFO", "name": "app", "message": "Data loaded from cache for page: crop_condition_and_progress", "pathname": "c:\\Users\\yuhang.hou\\projects\\multipage_dash\\olam_dash\\data\\data_loader3.py", "lineno": 129, "process": 16084, "threadName": "MainThread"}
  Progress: 65764 rows, types: ['silking' 'dented' 'harvested' 'dough' 'harvested (silage)' 'emerged'
 'mature' 'planted' 'milk' 'seedbed prepared']

--- SOYBEANS ---
{"asctime": "2026-04-09 17:22:47,177", "levelname": "INFO", "name": "app", "message": "Data loaded from cache for page: crop_condition_and_progress", "pathname": "c:\\Users\\yuha

In [4]:
price_data = {}
for cmd, info in COMMODITIES.items():
    print(f'Loading {info["name"]} price...')
    loader = PriceDataLoader('price_data', reload_freq=1)
    df = loader.load_data(info['price_cmd'], filter_holiday=True)
    df['date'] = pd.to_datetime(df['date'])
    df['month_code'] = df['expiry_month'].map(MONTH_CODE)
    df = df[df['month_code'] == info['contract_month']].reset_index(drop=True)
    df = df.sort_values(['date', 'expiry_year', 'expiry_month'])
    df = df.groupby('date').head(1).reset_index(drop=True)
    df['return'] = df['value'].pct_change()
    df['log_return'] = np.log(df['value'] / df['value'].shift(1))
    df['realized_vol_20d'] = df['log_return'].rolling(20, min_periods=10).std() * np.sqrt(252)
    price_data[cmd] = df
    print(f'  -> {len(df)} rows, {df["date"].min().date()} to {df["date"].max().date()}')

Loading Corn price...
  -> 6879 rows, 1998-12-02 to 2026-04-08
Loading Soybeans price...
  -> 6892 rows, 1998-11-12 to 2026-04-08


---
## 2. Custom Signal Engineering
6 new signals: Condition Momentum, Acceleration, YoY Change, Extreme Flags, G+E Z-score, G+E Momentum

In [5]:
def build_custom_signals(condition_df, lookback_years=10):
    df = condition_df[condition_df['location'] == 'US TOTAL'].copy()
    df['date'] = pd.to_datetime(df['date'])
    pivoted = df.pivot(index='date', columns='type', values='value').reset_index()
    for col in ['excellent', 'good', 'fair', 'poor', 'very poor']:
        if col not in pivoted.columns: pivoted[col] = 0
        pivoted[col] = pivoted[col].fillna(0)
    weights = {'excellent': 1.25, 'good': 1.0, 'fair': 0.75, 'poor': 0.5, 'very poor': 0.0}
    pivoted['condition_index'] = sum(pivoted[c]*w for c,w in weights.items())
    pivoted['good_excellent_pct'] = pivoted['excellent'] + pivoted['good']
    pivoted = pivoted.sort_values('date').reset_index(drop=True)
    pivoted['year'] = pivoted['date'].dt.year
    pivoted['week'] = pivoted['date'].dt.isocalendar().week.astype(int)
    # Signal 1: Momentum
    pivoted['condition_change'] = pivoted['condition_index'].diff()
    pivoted['condition_momentum_3w'] = pivoted['condition_change'].rolling(3, min_periods=2).mean()
    # Signal 2: Acceleration
    pivoted['condition_acceleration'] = pivoted['condition_change'].diff()
    # Signal 3: YoY
    prev = pivoted[['week','year','condition_index']].copy()
    prev['year'] = prev['year'] - 1
    prev = prev.rename(columns={'condition_index':'prev_yr_index'})
    pivoted = pivoted.merge(prev[['week','year','prev_yr_index']], on=['week','year'], how='left')
    pivoted['yoy_condition_change'] = pivoted['condition_index'] - pivoted['prev_yr_index']
    # Signal 4: Extreme flags
    pivoted['condition_pct_rank'] = pivoted['condition_index'].rank(pct=True)
    pivoted['is_extreme_good'] = (pivoted['condition_pct_rank'] >= 0.8).astype(int)
    pivoted['is_extreme_bad'] = (pivoted['condition_pct_rank'] <= 0.2).astype(int)
    # Signal 5: G+E Z-score
    hist = pivoted.copy()
    def deseason_ge(row):
        sw = hist[(hist['week']==row['week']) & (hist['year']<row['year']) & (hist['year']>row['year']-lookback_years)]
        if len(sw)<3: return np.nan
        m,s = sw['good_excellent_pct'].mean(), sw['good_excellent_pct'].std()
        return (row['good_excellent_pct']-m)/s if s>0 and not pd.isna(s) else np.nan
    pivoted['ge_zscore'] = pivoted.apply(deseason_ge, axis=1)
    # Signal 6: G+E Momentum
    pivoted['ge_change'] = pivoted['good_excellent_pct'].diff()
    pivoted['ge_momentum_3w'] = pivoted['ge_change'].rolling(3, min_periods=2).mean()
    pivoted['signal_date'] = pivoted['date'].apply(lambda x: apply_date_rules(x, '+2B', 'CBT'))
    pivoted['signal_date'] = pd.to_datetime(pivoted['signal_date'])
    cols = ['signal_date','condition_momentum_3w','condition_acceleration','yoy_condition_change',
            'is_extreme_good','is_extreme_bad','ge_zscore','ge_momentum_3w','good_excellent_pct','condition_index']
    return pivoted[cols]

custom_signals = {}
for cmd in COMMODITIES:
    print(f'Building {cmd} custom signals...')
    custom_signals[cmd] = build_custom_signals(raw_condition[cmd])
    print(f'  -> {len(custom_signals[cmd])} rows')
    display(custom_signals[cmd].tail(3))
custom_signals[cmd].describe()

Building CORN custom signals...
  -> 829 rows


type,signal_date,condition_momentum_3w,condition_acceleration,yoy_condition_change,is_extreme_good,is_extreme_bad,ge_zscore,ge_momentum_3w,good_excellent_pct,condition_index
826,2025-09-16,-0.833333,-0.50,NaN,0,0,0.855256,-1.333333,67.0,92.25
827,2025-09-23,-0.500000,0.25,NaN,0,0,0.664514,-1.000000,66.0,91.75
828,2025-09-30,-0.416667,0.50,NaN,0,0,0.688220,-0.666667,66.0,91.75


Building SOYBEANS custom signals...
  -> 734 rows


type,signal_date,condition_momentum_3w,condition_acceleration,yoy_condition_change,is_extreme_good,is_extreme_bad,ge_zscore,ge_momentum_3w,good_excellent_pct,condition_index
731,2025-09-16,-1.083333,-0.50,NaN,0,0,0.359330,-2.000000,63.0,89.75
732,2025-09-23,-0.750000,-0.50,NaN,0,0,0.061323,-1.333333,61.0,88.50
733,2025-09-30,-0.333333,2.25,NaN,0,0,0.125883,-0.666667,62.0,89.50


type,signal_date,condition_momentum_3w,condition_acceleration,yoy_condition_change,is_extreme_good,is_extreme_bad,ge_zscore,ge_momentum_3w,good_excellent_pct,condition_index
count,734,732.000000,732.000000,678.000000,734.000000,734.000000,660.000000,732.000000,734.000000,734.000000
mean,2006-08-27 11:28:36.621253376,-0.002220,0.002391,-0.016593,0.208447,0.198910,0.046977,0.000228,59.745232,88.446185
min,1986-06-24 00:00:00,-7.916667,-20.250000,-25.500000,0.000000,0.000000,-5.622998,-15.666667,16.000000,65.500000
25%,1996-10-02 18:00:00,-0.666667,-1.000000,-4.500000,0.000000,0.000000,-0.421587,-1.000000,55.000000,86.000000
50%,2006-09-22 12:00:00,0.000000,0.000000,0.000000,0.000000,0.000000,0.196374,0.000000,61.000000,89.500000
75%,2016-07-31 06:00:00,0.416667,1.000000,4.937500,0.000000,0.000000,0.757463,0.666667,67.000000,92.250000
max,2025-09-30 00:00:00,7.250000,18.500000,28.750000,1.000000,1.000000,2.886751,11.666667,83.000000,100.250000
std,NaN,1.460803,3.107360,8.374155,0.406475,0.399452,1.150212,2.622098,10.569420,5.810942


---
## 3. Master Analysis Table
Merge all signals with forward returns at multiple horizons (1W, 2W, 4W).

In [6]:
def get_season(month):
    for phase, months in SEASON_PHASES.items():
        if month in months: return phase
    return 'unknown'

def build_master_table(cmd, signals_df, custom_df, price_df):
    sig = signals_df.pivot_table(index='signal_date', columns='signal_name', values='signal_value').reset_index()
    sig['signal_date'] = pd.to_datetime(sig['signal_date'])
    merged = pd.merge(sig, custom_df, on='signal_date', how='left')
    prc = price_df[['date','value']].copy().sort_values('date').reset_index(drop=True)
    for h in FORWARD_HORIZONS:
        prc[f'fwd_{h}d_ret'] = prc['value'].shift(-h) / prc['value'] - 1
    prc['date'] = pd.to_datetime(prc['date'])
    merged = pd.merge(merged, prc, left_on='signal_date', right_on='date', how='inner', suffixes=('','_price'))
    merged = merged.dropna(subset=['fwd_5d_ret'])
    merged['month'] = merged['signal_date'].dt.month
    merged['year'] = merged['signal_date'].dt.year
    merged['season'] = merged['month'].apply(get_season)
    vol_df = price_df[['date','realized_vol_20d']].copy()
    vol_df['date'] = pd.to_datetime(vol_df['date'])
    merged = pd.merge(merged, vol_df, left_on='signal_date', right_on='date', how='left', suffixes=('','_vol'))
    if 'realized_vol_20d' in merged.columns:
        merged['realized_vol_20d'] = pd.to_numeric(merged['realized_vol_20d'], errors='coerce')
        vol_med = merged['realized_vol_20d'].median()
        merged['vol_regime'] = np.where(merged['realized_vol_20d'].isna(),'unknown',
            np.where(merged['realized_vol_20d']>vol_med,'high_vol','low_vol'))
    trend_df = price_df[['date','value']].copy().sort_values('date')
    trend_df['date'] = pd.to_datetime(trend_df['date'])
    trend_df['price_20d_momentum'] = trend_df['value']/trend_df['value'].shift(20)-1
    merged = pd.merge(merged, trend_df[['date','price_20d_momentum']], left_on='signal_date', right_on='date', how='left', suffixes=('','_trend'))
    if 'price_20d_momentum' in merged.columns:
        merged['price_20d_momentum'] = pd.to_numeric(merged['price_20d_momentum'], errors='coerce')
        merged['trend_regime'] = np.where(merged['price_20d_momentum'].isna(),'unknown',
            np.where(merged['price_20d_momentum']>0.02,'uptrend',
            np.where(merged['price_20d_momentum']<-0.02,'downtrend','neutral')))
    return merged

master = {}
for cmd in COMMODITIES:
    master[cmd] = build_master_table(cmd, all_signals[cmd], custom_signals[cmd], price_data[cmd])
    print(f'{cmd}: {len(master[cmd])} rows x {len(master[cmd].columns)} cols, {master[cmd]["signal_date"].min().date()} -> {master[cmd]["signal_date"].max().date()}')
    print(f'  Seasons: {master[cmd]["season"].value_counts().to_dict()}')
    display(master[cmd].tail(2))
master[cmd].describe()

CORN: 562 rows x 27 cols, 1999-06-02 -> 2025-09-30
  Seasons: {'growing': 355, 'harvest': 189, 'planting': 18}


,signal_date,crop_condition_historical_surprise,crop_condition_normalized,crop_condition_normalized_diff,condition_momentum_3w,condition_acceleration,yoy_condition_change,is_extreme_good,is_extreme_bad,ge_zscore,...,fwd_20d_ret,month,year,season,date_vol,realized_vol_20d,vol_regime,date_trend,price_20d_momentum,trend_regime
560,2025-09-23,-1.510780,0.804938,-0.155001,-0.500000,0.25,NaN,0.0,0.0,0.664514,...,-0.015249,9,2025,harvest,2025-09-23,0.180475,low_vol,2025-09-23,0.03396,uptrend
561,2025-09-30,0.354882,0.817595,0.012657,-0.416667,0.50,NaN,0.0,0.0,0.688220,...,0.039711,9,2025,harvest,2025-09-30,0.160758,low_vol,2025-09-30,-0.01773,neutral


SOYBEANS: 504 rows x 27 cols, 1999-06-15 -> 2025-09-30
  Seasons: {'growing': 332, 'harvest': 172}


,signal_date,crop_condition_historical_surprise,crop_condition_normalized,crop_condition_normalized_diff,condition_momentum_3w,condition_acceleration,yoy_condition_change,is_extreme_good,is_extreme_bad,ge_zscore,...,fwd_20d_ret,month,year,season,date_vol,realized_vol_20d,vol_regime,date_trend,price_20d_momentum,trend_regime
502,2025-09-23,-3.270206,0.034158,-0.343619,-0.750000,-0.50,NaN,0.0,0.0,0.061323,...,0.018528,9,2025,harvest,2025-09-23,0.117003,low_vol,2025-09-23,-0.034121,downtrend
503,2025-09-30,1.338209,0.215582,0.181424,-0.333333,2.25,NaN,0.0,0.0,0.125883,...,0.076366,9,2025,harvest,2025-09-30,0.110160,low_vol,2025-09-30,-0.037704,downtrend


,signal_date,crop_condition_historical_surprise,crop_condition_normalized,crop_condition_normalized_diff,condition_momentum_3w,condition_acceleration,yoy_condition_change,is_extreme_good,is_extreme_bad,ge_zscore,...,value,fwd_5d_ret,fwd_10d_ret,fwd_20d_ret,month,year,date_vol,realized_vol_20d,date_trend,price_20d_momentum
count,504,474.000000,504.000000,477.000000,479.000000,479.000000,445.000000,479.000000,479.000000,476.000000,...,504.000000,504.000000,504.000000,504.000000,504.000000,504.000000,504,504.000000,504,504.000000
mean,2012-09-29 15:14:17.142857216,-0.044773,-0.116765,-0.005568,-0.033751,-0.029749,-0.210674,0.223382,0.192067,-0.015994,...,949.247024,-0.000791,-0.001158,-0.001683,7.851190,2012.136905,2012-09-29 15:14:17.142857216,0.234681,2012-09-29 15:14:17.142857216,-0.004319
min,1999-06-15 00:00:00,-8.107584,-7.029892,-2.455221,-4.166667,-20.250000,-25.500000,0.000000,0.000000,-5.622998,...,436.750000,-0.119629,-0.219882,-0.235046,6.000000,1999.000000,1999-06-15 00:00:00,0.075172,1999-06-15 00:00:00,-0.246708
25%,2005-10-10 00:00:00,-0.661357,-0.574411,-0.151742,-0.666667,-0.750000,-4.250000,0.000000,0.000000,-0.580844,...,639.062500,-0.023424,-0.033456,-0.049605,7.000000,2005.000000,2005-10-10 00:00:00,0.170195,2005-10-10 00:00:00,-0.052477
50%,2012-09-01 00:00:00,0.044744,0.193303,-0.002800,-0.083333,0.000000,0.000000,0.000000,0.000000,0.221160,...,951.625000,-0.002277,-0.003694,-0.003537,8.000000,2012.000000,2012-09-01 00:00:00,0.212116,2012-09-01 00:00:00,-0.004920
75%,2019-07-31 18:00:00,0.764625,0.734521,0.162768,0.333333,1.000000,4.500000,0.000000,0.000000,0.788094,...,1189.062500,0.022414,0.030372,0.042775,9.000000,2019.000000,2019-07-31 18:00:00,0.280556,2019-07-31 18:00:00,0.040019
max,2025-09-30 00:00:00,4.765654,2.874831,3.312818,7.250000,18.500000,22.750000,1.000000,1.000000,2.886751,...,1747.500000,0.112729,0.162271,0.258337,10.000000,2025.000000,2025-09-30 00:00:00,0.574759,2025-09-30 00:00:00,0.189289
std,NaN,1.318728,1.365202,0.389828,1.386322,2.842129,7.175242,0.416948,0.394337,1.232506,...,316.342901,0.035298,0.050384,0.073323,1.281338,7.746423,NaN,0.087555,NaN,0.072533


In [7]:
KEY_SIGNALS = ['crop_condition_normalized','crop_condition_normalized_diff',
    'crop_condition_historical_surprise','condition_momentum_3w',
    'ge_zscore','yoy_condition_change','condition_acceleration']

---
## 4. Signal Quality Assessment
IC, ICIR, Hit Rate, Turnover per signal across multiple forward return horizons.

In [8]:
def compute_signal_quality(df, signal_col, return_col='fwd_5d_ret'):
    data = df[[signal_col, return_col, 'year']].dropna().copy()
    if len(data) < 30:
        return {'signal': signal_col, 'n': len(data), 'IC': np.nan, 'ICIR': np.nan, 'hit_rate': np.nan, 'turnover': np.nan}
    ic = data[signal_col].corr(data[return_col])
    yearly_ics = data.groupby('year').apply(lambda g: g[signal_col].corr(g[return_col])).dropna()
    icir = yearly_ics.mean()/yearly_ics.std() if len(yearly_ics)>=3 and yearly_ics.std()>0 else np.nan
    hit = (np.sign(data[signal_col]) == np.sign(data[return_col])).mean()
    signs = np.sign(data[signal_col].sort_index())
    turnover = (signs != signs.shift(1)).sum() / len(data)
    return {'signal': signal_col, 'n': len(data), 'IC': ic, 'ICIR': icir, 'hit_rate': hit, 'turnover': turnover}

def get_signal_columns(df):
    exclude = {'signal_date','date','value','year','month','week_of_year','season',
               'realized_vol_20d','vol_regime','price_20d_momentum','trend_regime','date_price','date_vol'}
    exclude |= {c for c in df.columns if 'fwd_' in c}
    return [c for c in df.columns if c not in exclude and df[c].dtype in ['float64','int64','Int64','float32','int32']]

for cmd in COMMODITIES:
    m = master[cmd]
    sig_cols = get_signal_columns(m)
    print(f'\n{"="*70}\n{COMMODITIES[cmd]["name"]} - Signal Quality\n{"="*70}')
    results = []
    for horizon in FORWARD_HORIZONS:
        for sig in sig_cols:
            r = compute_signal_quality(m, sig, f'fwd_{horizon}d_ret')
            r['horizon'] = f'{horizon}d'
            results.append(r)
    quality_df = pd.DataFrame(results).sort_values('IC', ascending=False, key=abs)
    top = quality_df[quality_df['horizon']=='5d'].head(10)
    print(f'\nTop signals by |IC| (5d fwd):')
    display(top.style.format({'IC':'{:.4f}','ICIR':'{:.2f}','hit_rate':'{:.1%}','turnover':'{:.1%}'}))
    # Bar chart
    fig = go.Figure()
    for horizon in FORWARD_HORIZONS:
        h_df = quality_df[quality_df['horizon']==f'{horizon}d'].sort_values('IC', key=abs)
        fig.add_trace(go.Bar(y=h_df['signal'], x=h_df['IC'], orientation='h', name=f'{horizon}d fwd'))
    fig.update_layout(title=f'{COMMODITIES[cmd]["name"]} - IC by Signal & Horizon',
        barmode='group', height=max(400,len(sig_cols)*25), template='plotly_white', xaxis_title='IC')
    fig.show()
    master[cmd+'_quality'] = quality_df


Corn - Signal Quality

Top signals by |IC| (5d fwd):


,signal,n,IC,ICIR,hit_rate,turnover,horizon
0,crop_condition_historical_surprise,532,-0.0885,-0.30,47.2%,44.7%,5d
6,is_extreme_good,537,-0.0723,-0.19,4.7%,6.3%,5d
2,crop_condition_normalized_diff,535,-0.0641,-0.17,47.1%,45.4%,5d
8,ge_zscore,533,-0.0312,0.26,46.9%,11.1%,5d
9,ge_momentum_3w,537,-0.0302,-0.15,41.7%,30.9%,5d
1,crop_condition_normalized,562,-0.0287,0.14,47.7%,12.8%,5d
7,is_extreme_bad,537,-0.0278,-0.56,10.2%,7.6%,5d
3,condition_momentum_3w,537,-0.0270,-0.13,44.5%,26.6%,5d
5,yoy_condition_change,497,0.0242,0.30,48.7%,8.0%,5d
4,condition_acceleration,537,0.0207,0.19,44.5%,68.0%,5d



Soybeans - Signal Quality

Top signals by |IC| (5d fwd):


,signal,n,IC,ICIR,hit_rate,turnover,horizon
1,crop_condition_normalized,504,-0.0793,0.22,51.8%,8.7%,5d
8,ge_zscore,476,-0.0791,0.33,50.2%,8.4%,5d
6,is_extreme_good,479,-0.0674,-0.34,10.2%,8.8%,5d
5,yoy_condition_change,445,-0.0656,0.15,46.7%,11.7%,5d
9,ge_momentum_3w,479,-0.0626,-0.18,43.6%,32.6%,5d
3,condition_momentum_3w,479,-0.0588,-0.14,45.9%,26.9%,5d
10,good_excellent_pct,479,-0.0440,0.36,46.1%,0.2%,5d
11,condition_index,479,-0.0400,0.36,46.1%,0.2%,5d
2,crop_condition_normalized_diff,477,0.0347,0.28,50.3%,43.8%,5d
0,crop_condition_historical_surprise,474,0.0278,0.21,46.4%,45.8%,5d


In [9]:
# Basket monotonicity check
from scipy.stats import spearmanr
for cmd in COMMODITIES:
    m = master[cmd]
    print(f'\n{COMMODITIES[cmd]["name"]} - Basket Analysis:')
    available = [s for s in KEY_SIGNALS if s in m.columns]
    basket_results = []
    for sig in available:
        data = m[[sig,'fwd_5d_ret']].dropna().copy()
        if len(data) < 25: continue
        data['basket'] = pd.qcut(data[sig], q=5, labels=False, duplicates='drop')
        stats = data.groupby('basket')['fwd_5d_ret'].agg(['mean','count']).reset_index()
        mono, _ = spearmanr(stats['basket'], stats['mean'])
        ls = stats['mean'].iloc[-1] - stats['mean'].iloc[0]
        basket_results.append({'signal':sig,'monotonicity':mono,'long_short':ls,'B1':stats['mean'].iloc[0],'B5':stats['mean'].iloc[-1]})
    bdf = pd.DataFrame(basket_results).sort_values('long_short', ascending=False, key=abs)
    display(bdf.style.format({'monotonicity':'{:.3f}','long_short':'{:.4f}','B1':'{:.4f}','B5':'{:.4f}'}))


Corn - Basket Analysis:


,signal,monotonicity,long_short,B1,B5
2,crop_condition_historical_surprise,-0.900,-0.0114,0.0035,-0.0080
5,yoy_condition_change,0.100,0.0069,-0.0056,0.0013
6,condition_acceleration,0.100,0.0050,-0.0037,0.0013
4,ge_zscore,0.000,-0.0044,0.0021,-0.0023
3,condition_momentum_3w,-0.600,-0.0036,-0.0024,-0.0060
1,crop_condition_normalized_diff,-0.700,-0.0036,-0.0015,-0.0051
0,crop_condition_normalized,-0.100,-0.0013,0.0015,0.0003



Soybeans - Basket Analysis:


,signal,monotonicity,long_short,B1,B5
4,ge_zscore,-0.400,-0.0082,0.0034,-0.0048
0,crop_condition_normalized,-0.700,-0.0067,0.0024,-0.0043
3,condition_momentum_3w,-0.200,-0.0057,0.0018,-0.0039
5,yoy_condition_change,-0.600,-0.0049,0.0012,-0.0037
6,condition_acceleration,0.700,0.0047,-0.0031,0.0016
1,crop_condition_normalized_diff,0.700,0.0019,-0.0042,-0.0023
2,crop_condition_historical_surprise,-0.300,-0.0001,-0.0020,-0.0021


---
## 5. Seasonal & Regime Analysis
IC broken down by crop season, volatility regime, and price trend regime.

In [10]:
for cmd in COMMODITIES:
    m = master[cmd]
    available = [s for s in KEY_SIGNALS if s in m.columns]
    print(f'\n{"="*70}\n{COMMODITIES[cmd]["name"]} - IC by Season & Regime\n{"="*70}')
    # IC by Season
    season_results = []
    for season in ['planting','growing','harvest','off_season']:
        subset = m[m['season']==season]
        for sig in available:
            data = subset[[sig,'fwd_5d_ret']].dropna()
            if len(data)>=10:
                season_results.append({'season':season,'signal':sig,'IC':data[sig].corr(data['fwd_5d_ret']),'n':len(data)})
    sdf = pd.DataFrame(season_results)
    if len(sdf)>0:
        pivot = sdf.pivot(index='signal',columns='season',values='IC')
        for s in ['planting','growing','harvest','off_season']:
            if s not in pivot.columns: pivot[s]=np.nan
        pivot = pivot[['planting','growing','harvest','off_season']]
        fig = go.Figure(go.Heatmap(z=pivot.values, x=pivot.columns, y=pivot.index,
            colorscale='RdYlGn', zmid=0,
            text=[[f'{v:.3f}' if not pd.isna(v) else '' for v in row] for row in pivot.values],
            texttemplate='%{text}', textfont=dict(size=10)))
        fig.update_layout(title=f'{COMMODITIES[cmd]["name"]} - IC by Signal x Season',
            height=max(400,len(available)*40), template='plotly_white')
        fig.show()
    # IC by Vol & Trend regime
    if 'vol_regime' in m.columns:
        print('\nIC by Vol Regime:')
        vol_res = []
        for regime in ['high_vol','low_vol']:
            for sig in available:
                data = m[m['vol_regime']==regime][[sig,'fwd_5d_ret']].dropna()
                if len(data)>=10: vol_res.append({'vol_regime':regime,'signal':sig,'IC':data[sig].corr(data['fwd_5d_ret'])})
        vdf = pd.DataFrame(vol_res).pivot(index='signal',columns='vol_regime',values='IC')
        display(vdf.style.format('{:.4f}'))
    if 'trend_regime' in m.columns:
        print('\nIC by Trend Regime:')
        trend_res = []
        for regime in ['uptrend','downtrend','neutral']:
            for sig in available:
                data = m[m['trend_regime']==regime][[sig,'fwd_5d_ret']].dropna()
                if len(data)>=10: trend_res.append({'trend_regime':regime,'signal':sig,'IC':data[sig].corr(data['fwd_5d_ret'])})
        tdf = pd.DataFrame(trend_res).pivot(index='signal',columns='trend_regime',values='IC')
        for c in ['uptrend','neutral','downtrend']:
            if c not in tdf.columns: tdf[c]=np.nan
        display(tdf[['uptrend','neutral','downtrend']].style.format('{:.4f}'))


Corn - IC by Season & Regime



IC by Vol Regime:


vol_regime,high_vol,low_vol
signal,,
condition_acceleration,0.0763,-0.0209
condition_momentum_3w,-0.0272,-0.0463
crop_condition_historical_surprise,-0.0626,-0.1475
crop_condition_normalized,-0.0527,-0.0000
crop_condition_normalized_diff,-0.0366,-0.1157
ge_zscore,-0.0525,-0.0074
yoy_condition_change,0.0373,-0.0069



IC by Trend Regime:


trend_regime,uptrend,neutral,downtrend
signal,,,
condition_acceleration,-0.0285,0.0878,0.0503
condition_momentum_3w,-0.0212,0.0965,-0.0542
crop_condition_historical_surprise,-0.0973,-0.0924,-0.0500
crop_condition_normalized,-0.0360,0.1461,-0.0464
crop_condition_normalized_diff,-0.0969,-0.0517,-0.0056
ge_zscore,-0.0403,0.1425,-0.0418
yoy_condition_change,0.0436,0.1048,-0.0095



Soybeans - IC by Season & Regime



IC by Vol Regime:


vol_regime,high_vol,low_vol
signal,,
condition_acceleration,0.0137,-0.0137
condition_momentum_3w,-0.0534,-0.0784
crop_condition_historical_surprise,0.0431,0.0034
crop_condition_normalized,-0.1119,-0.0040
crop_condition_normalized_diff,0.0865,-0.0407
ge_zscore,-0.1324,0.0527
yoy_condition_change,-0.0869,-0.0488



IC by Trend Regime:


trend_regime,uptrend,neutral,downtrend
signal,,,
condition_acceleration,0.0132,-0.0006,-0.0152
condition_momentum_3w,-0.0473,-0.1108,-0.0223
crop_condition_historical_surprise,0.0341,0.1181,0.0140
crop_condition_normalized,-0.0229,-0.2762,-0.0012
crop_condition_normalized_diff,0.0495,0.0955,0.0043
ge_zscore,-0.0445,-0.2464,-0.0011
yoy_condition_change,-0.1553,-0.0236,0.0442


---
## 6. IC Time-Series Heatmap & PnL Simulation
Rolling IC stability and simple long/short backtest for top signals.

In [11]:
for cmd in COMMODITIES:
    m = master[cmd].copy()
    available = [s for s in KEY_SIGNALS if s in m.columns]
    print(f'\n{COMMODITIES[cmd]["name"]} - IC by Year x Signal:')
    ic_year = []
    for yr in sorted(m['year'].unique()):
        for sig in available:
            data = m[m['year']==yr][[sig,'fwd_5d_ret']].dropna()
            if len(data)>=5: ic_year.append({'year':yr,'signal':sig,'IC':data[sig].corr(data['fwd_5d_ret'])})
    icdf = pd.DataFrame(ic_year).pivot(index='signal',columns='year',values='IC')
    fig = go.Figure(go.Heatmap(z=icdf.values, x=[str(y) for y in icdf.columns], y=icdf.index,
        colorscale='RdYlGn', zmid=0, zmin=-0.5, zmax=0.5,
        text=[[f'{v:.2f}' if not pd.isna(v) else '' for v in row] for row in icdf.values],
        texttemplate='%{text}', textfont=dict(size=9)))
    fig.update_layout(title=f'{COMMODITIES[cmd]["name"]} - IC by Year x Signal',
        height=max(400,len(available)*40), template='plotly_white')
    fig.show()


Corn - IC by Year x Signal:



Soybeans - IC by Year x Signal:


In [12]:
def simulate_pnl(df, signal_col, return_col='fwd_5d_ret', holding_period=5):
    data = df[['signal_date', signal_col, return_col]].dropna().sort_values('signal_date').copy()
    data['position'] = np.sign(data[signal_col])
    data['strategy_ret'] = data['position'] * data[return_col]
    data['cum_strategy'] = (1 + data['strategy_ret']).cumprod()
    data['cum_buy_hold'] = (1 + data[return_col]).cumprod()
    total = (1 + data['strategy_ret']).prod() - 1
    n_obs = len(data)
    ann_ret = (1+total)**(52/n_obs) - 1 if n_obs>0 else np.nan
    sharpe = data['strategy_ret'].mean()/data['strategy_ret'].std()*np.sqrt(52) if data['strategy_ret'].std()>0 else np.nan
    dd = (data['cum_strategy']/data['cum_strategy'].cummax()-1).min()
    return data, {'ann_ret':ann_ret,'sharpe':sharpe,'max_dd':dd,'n_trades':n_obs}
def simulate_pnl_sum(df, signal_col, return_col="fwd_5d_ret", holding_period=5):
    """Cumulative sum return (non-compounding)."""
    data = df[["signal_date", signal_col, return_col]].dropna().sort_values("signal_date").copy()
    ic = data[signal_col].corr(data[return_col])
    direction = 1 if ic >= 0 else -1
    data["position"] = direction * np.sign(data[signal_col])
    data["strategy_ret"] = data["position"] * data[return_col]
    data["cum_strategy"] = data["strategy_ret"].cumsum()
    data["cum_buy_hold"] = data[return_col].cumsum()
    n_obs = len(data)
    ann_ret = data["strategy_ret"].mean() * 52 if n_obs > 0 else np.nan
    sharpe = data["strategy_ret"].mean()/data["strategy_ret"].std()*np.sqrt(52) if data["strategy_ret"].std()>0 else np.nan
    dd = (data["cum_strategy"] - data["cum_strategy"].cummax()).min()
    return data, {"ann_ret": ann_ret, "sharpe": sharpe, "max_dd": dd, "n_trades": n_obs, "direction": direction, "IC": ic}
    


for cmd in COMMODITIES:
    m = master[cmd]
    print(f'\n{"="*70}\n{COMMODITIES[cmd]["name"]} - PnL Simulation\n{"="*70}')
    available = [s for s in KEY_SIGNALS if s in m.columns]
    pnl_summary = []
    for sig in available:
        result, stats = simulate_pnl_sum(m, sig)
        pnl_summary.append({'signal':sig,**stats})
        fig = go.Figure()
        fig.add_trace(go.Scatter(x=result['signal_date'],y=result['cum_strategy'],name='Strategy'))
        fig.add_trace(go.Scatter(x=result['signal_date'],y=result['cum_buy_hold'],name='Buy&Hold',line=dict(dash='dash')))
        fig.update_layout(title=f'{sig} - Cumulative Return',template='plotly_white',height=300)
        fig.show()
    psdf = pd.DataFrame(pnl_summary).sort_values('sharpe',ascending=False)
    print('\nPnL Summary:')
    display(psdf.style.format({'ann_ret':'{:.2%}','sharpe':'{:.2f}','max_dd':'{:.2%}'}))


Corn - PnL Simulation



PnL Summary:


,signal,ann_ret,sharpe,max_dd,n_trades,direction,IC
3,condition_momentum_3w,12.83%,0.44,-90.99%,537,-1,-0.026970
2,crop_condition_historical_surprise,8.92%,0.29,-74.36%,532,-1,-0.088512
5,yoy_condition_change,4.42%,0.14,-119.11%,497,1,0.024197
1,crop_condition_normalized_diff,3.51%,0.12,-86.44%,535,-1,-0.064072
4,ge_zscore,2.17%,0.07,-92.31%,533,-1,-0.031221
0,crop_condition_normalized,1.84%,0.06,-85.06%,562,-1,-0.028651
6,condition_acceleration,1.26%,0.04,-91.97%,537,1,0.020686



Soybeans - PnL Simulation



PnL Summary:


,signal,ann_ret,sharpe,max_dd,n_trades,direction,IC
1,crop_condition_normalized_diff,14.51%,0.57,-28.34%,477,1,0.034734
5,yoy_condition_change,11.48%,0.45,-101.63%,445,-1,-0.065615
3,condition_momentum_3w,7.07%,0.28,-49.70%,479,-1,-0.058802
4,ge_zscore,3.05%,0.12,-86.68%,476,-1,-0.079071
0,crop_condition_normalized,0.81%,0.03,-79.57%,504,-1,-0.079271
2,crop_condition_historical_surprise,-1.54%,-0.06,-88.64%,474,1,0.027841
6,condition_acceleration,-3.38%,-0.14,-102.28%,479,-1,-0.000372


In [13]:
print('PnL simulation done.')

PnL simulation done.


---
## 7. Multi-Commodity Summary & Recommendation
Cross-commodity signal ranking and final actionable recommendation.

In [14]:
print('Cross-Commodity Signal Comparison (5d forward):')
cross_summary = []
for cmd in COMMODITIES:
    qdf = master.get(cmd+'_quality')
    if qdf is not None:
        for _, row in qdf[qdf['horizon']=='5d'].iterrows():
            cross_summary.append({'commodity':COMMODITIES[cmd]['name'],'signal':row['signal'],
                'IC':row['IC'],'ICIR':row['ICIR'],'hit_rate':row['hit_rate'],'turnover':row['turnover'],'n':row['n']})
cdf = pd.DataFrame(cross_summary).sort_values('IC',ascending=False,key=abs)
display(cdf.head(20).style.format({'IC':'{:.4f}','ICIR':'{:.2f}','hit_rate':'{:.1%}','turnover':'{:.1%}'}))

# Heatmap
if len(cdf)>0:
    pivot = cdf.pivot_table(index='signal',columns='commodity',values='IC')
    fig = go.Figure(go.Heatmap(z=pivot.values, x=pivot.columns, y=pivot.index,
        colorscale='RdYlGn', zmid=0,
        text=[[f'{v:.3f}' if not pd.isna(v) else '' for v in row] for row in pivot.values],
        texttemplate='%{text}', textfont=dict(size=11)))
    fig.update_layout(title='Cross-Commodity IC Comparison (5d fwd)',template='plotly_white',height=400)
    fig.show()

Cross-Commodity Signal Comparison (5d forward):


,commodity,signal,IC,ICIR,hit_rate,turnover,n
0,Corn,crop_condition_historical_surprise,-0.0885,-0.30,47.2%,44.7%,532
12,Soybeans,crop_condition_normalized,-0.0793,0.22,51.8%,8.7%,504
13,Soybeans,ge_zscore,-0.0791,0.33,50.2%,8.4%,476
1,Corn,is_extreme_good,-0.0723,-0.19,4.7%,6.3%,537
14,Soybeans,is_extreme_good,-0.0674,-0.34,10.2%,8.8%,479
15,Soybeans,yoy_condition_change,-0.0656,0.15,46.7%,11.7%,445
2,Corn,crop_condition_normalized_diff,-0.0641,-0.17,47.1%,45.4%,535
16,Soybeans,ge_momentum_3w,-0.0626,-0.18,43.6%,32.6%,479
17,Soybeans,condition_momentum_3w,-0.0588,-0.14,45.9%,26.9%,479
18,Soybeans,good_excellent_pct,-0.0440,0.36,46.1%,0.2%,479


In [15]:
print('\n' + '='*70)
print('RECOMMENDATION FRAMEWORK')
print('='*70)
print('''
Signal Selection Criteria:
  1. |IC| > 0.05 and consistent across years (ICIR > 0.5)
  2. Hit rate > 52% (better than random)
  3. Monotonic basket returns (monotonicity > 0.5)
  4. Stronger in growing season (Jun-Aug) when condition data is most informative
  5. Low turnover (< 30%) for implementability

Implementation Notes:
  - Use signal_date (next CBT business day after USDA release) for execution
  - Scale position by signal z-score for risk management
  - Combine top 2-3 uncorrelated signals via equal-weight composite
  - Increase position size during growing season, reduce in off-season
  - Monitor for structural breaks when USDA methodology changes

Next Steps:
  - Validate with out-of-sample data (most recent 2 years)
  - Test with continuous front-month contract for realistic execution
  - Add Soybean planting/harvest progress signals when available
  - Consider cross-commodity signal (corn-soy condition spread)
''')


RECOMMENDATION FRAMEWORK

Signal Selection Criteria:
  1. |IC| > 0.05 and consistent across years (ICIR > 0.5)
  2. Hit rate > 52% (better than random)
  3. Monotonic basket returns (monotonicity > 0.5)
  4. Stronger in growing season (Jun-Aug) when condition data is most informative
  5. Low turnover (< 30%) for implementability

Implementation Notes:
  - Use signal_date (next CBT business day after USDA release) for execution
  - Scale position by signal z-score for risk management
  - Combine top 2-3 uncorrelated signals via equal-weight composite
  - Increase position size during growing season, reduce in off-season
  - Monitor for structural breaks when USDA methodology changes

Next Steps:
  - Validate with out-of-sample data (most recent 2 years)
  - Test with continuous front-month contract for realistic execution
  - Add Soybean planting/harvest progress signals when available
  - Consider cross-commodity signal (corn-soy condition spread)



In [16]:
print('\n' + '='*70)
print('CURRENT SIGNAL SNAPSHOT')
print('='*70)
for cmd in COMMODITIES:
    print(f'\n{COMMODITIES[cmd]["name"]}:')
    cs = custom_signals[cmd]
    latest = cs.sort_values('signal_date').iloc[-1]
    print(f'  Latest signal date: {latest["signal_date"].date()}')
    print(f'  Good+Excellent: {latest["good_excellent_pct"]:.1f}%')
    print(f'  Condition Index: {latest["condition_index"]:.2f}')
    print(f'  GE Z-Score: {latest["ge_zscore"]:.2f}' if not pd.isna(latest['ge_zscore']) else '  GE Z-Score: N/A')
    print(f'  YoY Change: {latest["yoy_condition_change"]:.2f}' if not pd.isna(latest['yoy_condition_change']) else '  YoY Change: N/A')
    print(f'  3w Momentum: {latest["condition_momentum_3w"]:.3f}' if not pd.isna(latest['condition_momentum_3w']) else '  3w Momentum: N/A')
    if latest['is_extreme_good']: print('  *** EXTREME GOOD condition ***')
    if latest['is_extreme_bad']: print('  *** EXTREME BAD condition ***')


CURRENT SIGNAL SNAPSHOT

Corn:
  Latest signal date: 2025-09-30
  Good+Excellent: 66.0%
  Condition Index: 91.75
  GE Z-Score: 0.69
  YoY Change: N/A
  3w Momentum: -0.417

Soybeans:
  Latest signal date: 2025-09-30
  Good+Excellent: 62.0%
  Condition Index: 89.50
  GE Z-Score: 0.13
  YoY Change: N/A
  3w Momentum: -0.333


---
## 8. Additional Signals & Feature Engineering

In [17]:
def build_extended_signals(master_dict):
    for cmd in master_dict:
        if cmd in [ 'CORN_quality', 'SOYBEANS_quality', 'WINTER WHEAT_quality']:
            continue
        m = master_dict[cmd].copy().sort_values('signal_date').reset_index(drop=True)
        if 'crop_condition_historical_surprise' in m.columns and 'condition_momentum_3w' in m.columns:
            m['surprise_x_momentum'] = m['crop_condition_historical_surprise'] * m['condition_momentum_3w']
        if 'ge_zscore' in m.columns and 'condition_acceleration' in m.columns:
            m['zscore_x_accel'] = m['ge_zscore'] * m['condition_acceleration']
        if 'crop_condition_normalized' in m.columns and 'condition_momentum_3w' in m.columns:
            m['level_x_momentum'] = m['crop_condition_normalized'] * m['condition_momentum_3w']
        if 'yoy_condition_change' in m.columns and 'ge_zscore' in m.columns:
            m['yoy_x_zscore'] = m['yoy_condition_change'] * m['ge_zscore']
        for col in [c for c in m.columns if c in ['crop_condition_historical_surprise','condition_momentum_3w','ge_zscore']]:
            m[f'{col}_lag1'] = m[col].shift(1)
            m[f'{col}_lag2'] = m[col].shift(2)
        if 'crop_condition_historical_surprise' in m.columns:
            m['surprise_diff'] = m['crop_condition_historical_surprise'].diff()
        if 'realized_vol_20d' in m.columns and 'crop_condition_normalized_diff' in m.columns:
            m['vol_adj_diff'] = m['crop_condition_normalized_diff'] / m['realized_vol_20d'].replace(0, np.nan)
        if 'season' in m.columns and 'ge_zscore' in m.columns:
            for season in ['growing','harvest','planting']:
                m[f'ge_zscore_{season}'] = m['ge_zscore'] * (m['season'] == season).astype(int)
        if 'is_extreme_bad' in m.columns and 'condition_momentum_3w' in m.columns:
            m['extreme_bad_deteriorating'] = m['is_extreme_bad'] * (m['condition_momentum_3w'] < 0).astype(int)
        if 'is_extreme_good' in m.columns and 'condition_momentum_3w' in m.columns:
            m['extreme_good_improving'] = m['is_extreme_good'] * (m['condition_momentum_3w'] > 0).astype(int)
        master_dict[cmd] = m
        print(f'{cmd}: added extended signals -> {m.shape[1]} columns')
    return master_dict

master = build_extended_signals(master)

CORN: added extended signals -> 44 columns
SOYBEANS: added extended signals -> 44 columns


In [18]:
print("Module 8 done.")

Module 8 done.


---
## 9. Signal Correlation Analysis
Understand inter-signal correlations to identify redundant and complementary signals.

In [19]:
def get_all_signal_cols(df):
    exclude = {"signal_date","date","value","year","month","week_of_year","season",
               "realized_vol_20d","vol_regime","price_20d_momentum","trend_regime",
               "date_price","date_vol","good_excellent_pct","condition_index"}
    exclude |= {c for c in df.columns if "fwd_" in c}
    return [c for c in df.columns if c not in exclude and df[c].dtype in ["float64","int64","Int64","float32","int32"]]

for cmd in COMMODITIES:
    m = master[cmd]
    sig_cols = get_all_signal_cols(m)
    corr_matrix = m[sig_cols].corr()
    fig = go.Figure(go.Heatmap(z=corr_matrix.values, x=corr_matrix.columns, y=corr_matrix.index,
        colorscale="RdBu_r", zmid=0, zmin=-1, zmax=1,
        text=[[f"{v:.2f}" for v in row] for row in corr_matrix.values],
        texttemplate="%{text}", textfont=dict(size=7)))
    fig.update_layout(title=f"{cmd} - Signal Correlation", width=900, height=800, template="plotly_white")
    fig.show()
    print(f"Highly correlated pairs (|corr| > 0.6):")
    for ii in range(len(corr_matrix)):
        for jj in range(ii+1, len(corr_matrix)):
            val = corr_matrix.iloc[ii,jj]
            if abs(val) > 0.6:
                print(f"  {corr_matrix.index[ii]:40s} <-> {corr_matrix.columns[jj]:40s}: {val:.3f}")
    print(f"\nLow correlation pairs (|corr| < 0.3) with highest avg |IC|:")
    low_pairs = []
    for ii in range(len(corr_matrix)):
        for jj in range(ii+1, len(corr_matrix)):
            val = corr_matrix.iloc[ii,jj]
            if abs(val) < 0.3:
                s1, s2 = corr_matrix.index[ii], corr_matrix.columns[jj]
                ic1 = m[s1].corr(m["fwd_5d_ret"]) if s1 in m.columns else 0
                ic2 = m[s2].corr(m["fwd_5d_ret"]) if s2 in m.columns else 0
                low_pairs.append({"sig1":s1,"sig2":s2,"corr":val,"ic1":ic1,"ic2":ic2,"avg_ic":(abs(ic1)+abs(ic2))/2})
    if low_pairs:
        ldf = pd.DataFrame(low_pairs).sort_values("avg_ic",ascending=False).head(10)
        display(ldf.style.format({"corr":"{:.3f}","ic1":"{:.4f}","ic2":"{:.4f}","avg_ic":"{:.4f}"}))

Highly correlated pairs (|corr| > 0.6):
  crop_condition_historical_surprise       <-> surprise_diff                           : 0.699
  crop_condition_normalized                <-> yoy_condition_change                    : 0.615
  crop_condition_normalized                <-> is_extreme_bad                          : -0.644
  crop_condition_normalized                <-> ge_zscore                               : 0.985
  crop_condition_normalized                <-> yoy_x_zscore                            : -0.699
  crop_condition_normalized                <-> ge_zscore_lag1                          : 0.895
  crop_condition_normalized                <-> ge_zscore_lag2                          : 0.786
  crop_condition_normalized                <-> ge_zscore_growing                       : 0.767
  crop_condition_normalized_diff           <-> vol_adj_diff                            : 0.971
  condition_momentum_3w                    <-> ge_momentum_3w                          : 0.973
  condit

,sig1,sig2,corr,ic1,ic2,avg_ic
12,crop_condition_historical_surprise,condition_momentum_3w_lag1,0.093,-0.0885,-0.1076,0.0980
13,crop_condition_historical_surprise,condition_momentum_3w_lag2,0.058,-0.0885,-0.1018,0.0952
196,condition_momentum_3w_lag1,extreme_good_improving,0.182,-0.1076,-0.0729,0.0902
102,is_extreme_good,condition_momentum_3w_lag1,0.185,-0.0723,-0.1076,0.0899
192,condition_momentum_3w_lag1,surprise_diff,-0.116,-0.1076,-0.0715,0.0895
194,condition_momentum_3w_lag1,ge_zscore_harvest,0.039,-0.1076,0.0685,0.0880
201,condition_momentum_3w_lag2,extreme_good_improving,0.170,-0.1018,-0.0729,0.0874
17,crop_condition_historical_surprise,ge_zscore_growing,0.268,-0.0885,-0.0858,0.0871
103,is_extreme_good,condition_momentum_3w_lag2,0.191,-0.0723,-0.1018,0.0871
197,condition_momentum_3w_lag2,surprise_diff,-0.008,-0.1018,-0.0715,0.0866


Highly correlated pairs (|corr| > 0.6):
  crop_condition_historical_surprise       <-> crop_condition_normalized_diff          : 0.633
  crop_condition_historical_surprise       <-> surprise_diff                           : 0.640
  crop_condition_normalized                <-> yoy_condition_change                    : 0.643
  crop_condition_normalized                <-> is_extreme_bad                          : -0.622
  crop_condition_normalized                <-> ge_zscore                               : 0.986
  crop_condition_normalized                <-> yoy_x_zscore                            : -0.700
  crop_condition_normalized                <-> ge_zscore_lag1                          : 0.922
  crop_condition_normalized                <-> ge_zscore_lag2                          : 0.865
  crop_condition_normalized                <-> ge_zscore_growing                       : 0.847
  crop_condition_normalized_diff           <-> vol_adj_diff                            : 0.956
  condit

,sig1,sig2,corr,ic1,ic2,avg_ic
151,yoy_x_zscore,condition_momentum_3w_lag2,-0.277,0.0859,-0.0855,0.0857
184,surprise_diff,ge_zscore_growing,-0.038,0.0689,-0.0976,0.0833
150,yoy_x_zscore,condition_momentum_3w_lag1,-0.261,0.0859,-0.0771,0.0815
152,yoy_x_zscore,surprise_diff,0.035,0.0859,0.0689,0.0774
175,condition_momentum_3w_lag2,surprise_diff,-0.058,-0.0855,0.0689,0.0772
87,is_extreme_good,yoy_x_zscore,-0.004,-0.0674,0.0859,0.0767
90,is_extreme_good,condition_momentum_3w_lag2,0.247,-0.0674,-0.0855,0.0765
156,yoy_x_zscore,extreme_good_improving,0.002,0.0859,-0.0662,0.0761
178,condition_momentum_3w_lag2,extreme_good_improving,0.203,-0.0855,-0.0662,0.0758
79,yoy_condition_change,condition_momentum_3w_lag2,0.284,-0.0656,-0.0855,0.0755


---
## 10. Composite Signal Building
Build optimized composite signals: equal-weight, IC-weighted, and rolling-IC adaptive.

In [20]:
for cmd in COMMODITIES:
    m = master[cmd].copy().sort_values("signal_date").reset_index(drop=True)
    sig_cols = get_all_signal_cols(m)
    # Select uncorrelated signals with decent IC
    sig_ics = {}
    for s in sig_cols:
        valid = m[s].notna() & m["fwd_5d_ret"].notna()
        if valid.sum() > 30:
            sig_ics[s] = abs(m.loc[valid, s].corr(m.loc[valid, "fwd_5d_ret"]))
    candidates = sorted(sig_ics, key=sig_ics.get, reverse=True)[:10]
    # Remove highly correlated duplicates
    corr = m[candidates].corr().abs()
    selected = []
    for s in candidates:
        if not any(abs(corr.loc[s, x]) > 0.7 for x in selected if x in corr.columns):
            selected.append(s)
        if len(selected) >= 6: break
    print(f"\n{cmd} - Selected {len(selected)} uncorrelated signals:")
    for s in selected:
        ic_val = m[s].corr(m["fwd_5d_ret"])
        print(f"  {s:45s} IC={ic_val:.4f}")
    # Z-score standardize
    for s in selected:
        m[f"{s}_z"] = (m[s] - m[s].mean()) / m[s].std()
    z_cols = [f"{s}_z" for s in selected]
    # Method 1: Equal Weight
    m["composite_equal"] = m[z_cols].mean(axis=1)
    # Method 2: IC-Weighted
    ic_w = {}
    for s in selected:
        ic = m[s].corr(m["fwd_5d_ret"])
        ic_w[s] = ic if not pd.isna(ic) else 0
    total_ic = sum(abs(v) for v in ic_w.values())
    if total_ic > 0:
        ic_w = {k: abs(v)/total_ic * np.sign(v) for k,v in ic_w.items()}
    m["composite_ic_weighted"] = sum(m[f"{s}_z"] * w for s,w in ic_w.items())
    # Method 3: Rolling IC-Weighted (adaptive)
    for s in selected:
        m[f"{s}_ric"] = m[s].rolling(22, min_periods=20).corr(m["fwd_5d_ret"])
    ric_vals = []
    for idx in range(len(m)):
        rics = {s: m.iloc[idx][f"{s}_ric"] for s in selected}
        rics = {k: v for k,v in rics.items() if not pd.isna(v)}
        tot = sum(abs(v) for v in rics.values())
        if tot > 0:
            ws = {k: abs(v)/tot * np.sign(v) for k,v in rics.items()}
            val = sum(m.iloc[idx][f"{s}_z"] * w for s,w in ws.items() if f"{s}_z" in m.columns)
        else:
            val = np.nan
        ric_vals.append(val)
    m["composite_rolling_ic"] = ric_vals
    # Evaluate composites
    comp_names = ["composite_equal","composite_ic_weighted","composite_rolling_ic"]
    print(f"\nComposite Signal Quality:")
    comp_results = []
    for cn in comp_names:
        for h in FORWARD_HORIZONS:
            r = compute_signal_quality(m, cn, f"fwd_{h}d_ret")
            r["horizon"] = f"{h}d"
            comp_results.append(r)
    cdf = pd.DataFrame(comp_results).sort_values("IC", ascending=False, key=abs)
    display(cdf.style.format({"IC":"{:.4f}","ICIR":"{:.2f}","hit_rate":"{:.1%}","turnover":"{:.1%}"}))
    # PnL for best composite
    best = cdf.iloc[0]["signal"]
    result, stats = simulate_pnl_sum(m, best)
    print(f"Best: {best} -> Sharpe={stats['sharpe']}, AnnRet={stats['ann_ret']}, MaxDD={stats['max_dd']:.2%}")
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=result["signal_date"], y=result["cum_strategy"], name="Composite"))
    fig.add_trace(go.Scatter(x=result["signal_date"], y=result["cum_buy_hold"], name="Buy&Hold", line=dict(dash="dash")))
    fig.update_layout(title=f"{cmd} {best} - Cumulative Return", template="plotly_white", height=800)
    fig.show()
    master[cmd] = m


CORN - Selected 6 uncorrelated signals:
  condition_momentum_3w_lag1                    IC=-0.1076
  crop_condition_historical_surprise            IC=-0.0885
  ge_zscore_growing                             IC=-0.0858
  extreme_good_improving                        IC=-0.0729
  surprise_diff                                 IC=-0.0715
  ge_zscore_harvest                             IC=0.0685

Composite Signal Quality:


,signal,n,IC,ICIR,hit_rate,turnover,horizon
6,composite_rolling_ic,472,0.3779,2.26,63.1%,32.8%,5d
7,composite_rolling_ic,472,0.2878,1.15,60.6%,32.8%,10d
8,composite_rolling_ic,472,0.2405,0.57,60.6%,32.8%,20d
4,composite_ic_weighted,459,0.1811,0.57,58.0%,32.7%,10d
3,composite_ic_weighted,459,0.1644,0.69,57.3%,32.7%,5d
5,composite_ic_weighted,459,0.1483,0.35,55.3%,32.7%,20d
0,composite_equal,562,-0.1056,-0.40,44.5%,34.7%,5d
1,composite_equal,562,-0.0911,-0.23,46.3%,34.7%,10d
2,composite_equal,562,-0.0842,-0.13,49.1%,34.7%,20d


Best: composite_rolling_ic -> Sharpe=2.3853959118816346, AnnRet=0.7015529673991419, MaxDD=-22.26%



SOYBEANS - Selected 6 uncorrelated signals:
  ge_zscore_growing                             IC=-0.0976
  yoy_x_zscore                                  IC=0.0859
  condition_momentum_3w_lag2                    IC=-0.0855
  surprise_diff                                 IC=0.0689
  is_extreme_good                               IC=-0.0674
  yoy_condition_change                          IC=-0.0656

Composite Signal Quality:


,signal,n,IC,ICIR,hit_rate,turnover,horizon
6,composite_rolling_ic,438,0.3804,1.62,63.5%,31.1%,5d
7,composite_rolling_ic,438,0.3374,1.07,60.5%,31.1%,10d
8,composite_rolling_ic,438,0.2941,0.52,57.5%,31.1%,20d
4,composite_ic_weighted,397,0.1508,0.27,53.4%,21.2%,10d
1,composite_equal,504,-0.1321,-0.25,47.6%,26.0%,10d
5,composite_ic_weighted,397,0.1318,0.12,50.4%,21.2%,20d
2,composite_equal,504,-0.1255,-0.18,50.0%,26.0%,20d
3,composite_ic_weighted,397,0.1185,0.33,49.6%,21.2%,5d
0,composite_equal,504,-0.0635,-0.08,51.4%,26.0%,5d


Best: composite_rolling_ic -> Sharpe=2.5632416506297218, AnnRet=0.6245198889511697, MaxDD=-13.07%


---
## 11. Grid Search - Best Signal Combinations
Try all combinations of 2-4 signals to find the best composite.

In [21]:
from itertools import combinations

for cmd in COMMODITIES:
    m = master[cmd]
    base_sigs = [s for s in KEY_SIGNALS if s in m.columns]
    extra = ["surprise_x_momentum", "zscore_x_accel", "level_x_momentum",
             "yoy_x_zscore", "vol_adj_diff"]
    extra = [s for s in extra if s in m.columns]
    candidates = base_sigs + extra
    candidates = [s for s in candidates if m[s].notna().sum() > 50]
    # Standardize
    for s in candidates:
        m[f"{s}_z"] = (m[s] - m[s].mean()) / m[s].std()
    print(f"\n{cmd} - Grid Search over {len(candidates)} candidates")
    best_results = []
    for n_combo in [2, 3, 4]:
        for combo in combinations(candidates, n_combo):
            z_cols = [f"{s}_z" for s in combo]
            composite = m[z_cols].mean(axis=1)
            valid = composite.notna() & m["fwd_5d_ret"].notna()
            if valid.sum() < 50: continue
            ic5 = composite[valid].corr(m.loc[valid, "fwd_5d_ret"])
            ic10 = composite[valid].corr(m.loc[valid, "fwd_10d_ret"]) if "fwd_10d_ret" in m.columns else np.nan
            hit = (np.sign(composite[valid]) == np.sign(m.loc[valid, "fwd_5d_ret"])).mean()
            best_results.append({"signals": " + ".join(combo), "n": n_combo,
                "IC_5d": ic5, "IC_10d": ic10, "hit_rate": hit, "n_obs": valid.sum()})
    gdf = pd.DataFrame(best_results).sort_values("IC_5d", ascending=False, key=abs)
    print(f"\nTop 10 combinations by |IC| (5d):")
    display(gdf.head(10).style.format({"IC_5d":"{:.4f}","IC_10d":"{:.4f}","hit_rate":"{:.1%}"}))
    # PnL for top combo
    top_sigs = gdf.iloc[0]["signals"].split(" + ")
    top_z = [f"{s}_z" for s in top_sigs if f"{s}_z" in m.columns]
    m["best_grid_composite"] = m[top_z].mean(axis=1)
    result, stats = simulate_pnl_sum(m, "best_grid_composite")
    print(f"Best Grid Combo: Sharpe={stats['sharpe']:.2f}, AnnRet={stats['ann_ret']:.2%}")
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=result["signal_date"], y=result["cum_strategy"], name="Grid Strategy"))
    fig.add_trace(go.Scatter(x=result["signal_date"], y=result["cum_buy_hold"], name="Buy&Hold", line=dict(dash="dash")))
    fig.update_layout(title=f"{cmd} Best Grid Composite - Cumulative Return", template="plotly_white", height=400)
    fig.show()
    master[cmd] = m


CORN - Grid Search over 12 candidates

Top 10 combinations by |IC| (5d):


,signals,n,IC_5d,IC_10d,hit_rate,n_obs
11,crop_condition_normalized_diff + crop_condition_historical_surprise,2,-0.0941,-0.0871,45.8%,535
122,crop_condition_normalized_diff + crop_condition_historical_surprise + ge_zscore,3,-0.0919,-0.0904,45.7%,560
180,crop_condition_historical_surprise + ge_zscore + vol_adj_diff,3,-0.0901,-0.0783,45.0%,560
29,crop_condition_historical_surprise + vol_adj_diff,2,-0.0900,-0.0692,45.8%,535
465,crop_condition_normalized_diff + crop_condition_historical_surprise + ge_zscore + vol_adj_diff,4,-0.0861,-0.0901,47.0%,560
129,crop_condition_normalized_diff + crop_condition_historical_surprise + vol_adj_diff,3,-0.0846,-0.0863,46.2%,535
66,crop_condition_normalized + crop_condition_normalized_diff + crop_condition_historical_surprise,3,-0.0834,-0.0785,46.6%,562
750,yoy_condition_change + condition_acceleration + zscore_x_accel + level_x_momentum,4,0.0830,0.1005,51.2%,537
464,crop_condition_normalized_diff + crop_condition_historical_surprise + ge_zscore + yoy_x_zscore,4,-0.0828,-0.0742,46.4%,560
84,crop_condition_normalized + crop_condition_historical_surprise + vol_adj_diff,3,-0.0820,-0.0670,45.6%,562


Best Grid Combo: Sharpe=0.40, AnnRet=12.17%



SOYBEANS - Grid Search over 12 candidates

Top 10 combinations by |IC| (5d):


,signals,n,IC_5d,IC_10d,hit_rate,n_obs
194,crop_condition_historical_surprise + surprise_x_momentum + yoy_x_zscore,3,0.0905,0.0974,51.0%,492
658,condition_momentum_3w + ge_zscore + yoy_condition_change + level_x_momentum,4,-0.0881,-0.1310,48.0%,479
233,ge_zscore + yoy_condition_change + level_x_momentum,3,-0.0845,-0.1015,48.9%,479
720,ge_zscore + yoy_condition_change + zscore_x_accel + level_x_momentum,4,-0.0833,-0.1035,49.3%,479
202,condition_momentum_3w + ge_zscore + yoy_condition_change,3,-0.0830,-0.1389,49.1%,479
58,surprise_x_momentum + yoy_x_zscore,2,0.0827,0.1122,53.5%,467
657,condition_momentum_3w + ge_zscore + yoy_condition_change + zscore_x_accel,4,-0.0822,-0.1431,49.1%,479
30,condition_momentum_3w + ge_zscore,2,-0.0817,-0.1449,49.1%,479
38,ge_zscore + yoy_condition_change,2,-0.0815,-0.1214,48.7%,478
655,condition_momentum_3w + ge_zscore + yoy_condition_change + condition_acceleration,4,-0.0812,-0.1262,50.1%,479


Best Grid Combo: Sharpe=0.32, AnnRet=7.98%


---
## 12. Walk-Forward Lasso Regression (Out-of-Sample)
Rolling regression with Lasso for automatic feature selection and OOS validation.

In [22]:
from sklearn.linear_model import LassoCV
from sklearn.preprocessing import StandardScaler

for cmd in COMMODITIES:
    m = master[cmd].copy().sort_values("signal_date").reset_index(drop=True)
    features = [s for s in KEY_SIGNALS if s in m.columns]
    extra = ["surprise_x_momentum", "zscore_x_accel", "vol_adj_diff"]
    features += [s for s in extra if s in m.columns]
    print(f"\n{cmd} - Walk-Forward Lasso, features: {features}")
    data = m[["signal_date","year"] + features + ["fwd_5d_ret"]].dropna().copy()
    if len(data) < 100:
        print("Not enough data"); continue
    all_years = sorted(data["year"].unique())
    oos_preds = []
    for yr in all_years:
        yr_idx = all_years.index(yr)
        if yr_idx < 8: continue
        train = data[data["year"] < yr]
        test = data[data["year"] == yr]
        if len(train) < 50 or len(test) < 5: continue
        scaler = StandardScaler()
        X_train = scaler.fit_transform(train[features])
        y_train = train["fwd_5d_ret"].values
        X_test = scaler.transform(test[features])
        try:
            model = LassoCV(cv=min(5, len(train)//20), max_iter=5000, random_state=42)
            model.fit(X_train, y_train)
            preds = model.predict(X_test)
            coefs = {f: c for f, c in zip(features, model.coef_) if abs(c) > 1e-6}
            if coefs: print(f"  {yr}: non-zero coefs = {coefs}")
            for j, idx in enumerate(test.index):
                oos_preds.append({"idx": idx, "pred": preds[j], "actual": test.loc[idx, "fwd_5d_ret"],
                                  "signal_date": test.loc[idx, "signal_date"]})
        except Exception as e:
            print(f"  {yr}: failed - {e}")
    if not oos_preds: print("No OOS predictions"); continue
    oos_df = pd.DataFrame(oos_preds).set_index("idx")
    m["ml_prediction"] = np.nan
    m.loc[oos_df.index, "ml_prediction"] = oos_df["pred"]
    valid = m["ml_prediction"].notna() & m["fwd_5d_ret"].notna()
    if valid.sum() > 20:
        oos_ic = m.loc[valid, "ml_prediction"].corr(m.loc[valid, "fwd_5d_ret"])
        oos_hit = (np.sign(m.loc[valid, "ml_prediction"]) == np.sign(m.loc[valid, "fwd_5d_ret"])).mean()
        print(f"\nOOS Results: IC={oos_ic:.4f}, Hit={oos_hit:.1%}, N={valid.sum()}")
        result, stats = simulate_pnl_sum(m[valid].copy().reset_index(drop=True), "ml_prediction")
        print(f"  Sharpe={stats['sharpe']:.2f}, AnnRet={stats['ann_ret']:.2%}, MaxDD={stats['max_dd']:.2%}")
        fig = go.Figure()
        fig.add_trace(go.Scatter(x=result["signal_date"], y=result["cum_strategy"], name="Lasso OOS"))
        fig.add_trace(go.Scatter(x=result["signal_date"], y=result["cum_buy_hold"], name="Buy&Hold", line=dict(dash="dash")))
        fig.update_layout(title=f"{cmd} Walk-Forward Lasso (OOS)", template="plotly_white", height=400)
        fig.show()
    master[cmd] = m


CORN - Walk-Forward Lasso, features: ['crop_condition_normalized', 'crop_condition_normalized_diff', 'crop_condition_historical_surprise', 'condition_momentum_3w', 'ge_zscore', 'yoy_condition_change', 'condition_acceleration', 'surprise_x_momentum', 'zscore_x_accel', 'vol_adj_diff']
  2013: non-zero coefs = {'crop_condition_normalized_diff': np.float64(-0.0020164619408178635), 'crop_condition_historical_surprise': np.float64(-0.002567000363974592), 'yoy_condition_change': np.float64(0.0009441386204496292), 'condition_acceleration': np.float64(0.0007187285808178448), 'surprise_x_momentum': np.float64(0.003569893608678791), 'zscore_x_accel': np.float64(0.0016473698820207035)}
  2014: non-zero coefs = {'crop_condition_normalized_diff': np.float64(-0.0019751560997694835), 'crop_condition_historical_surprise': np.float64(-0.0035096579528666842), 'condition_momentum_3w': np.float64(-0.00044924324917225113), 'yoy_condition_change': np.float64(0.0015548856063142618), 'condition_acceleration':


SOYBEANS - Walk-Forward Lasso, features: ['crop_condition_normalized', 'crop_condition_normalized_diff', 'crop_condition_historical_surprise', 'condition_momentum_3w', 'ge_zscore', 'yoy_condition_change', 'condition_acceleration', 'surprise_x_momentum', 'zscore_x_accel', 'vol_adj_diff']
  2007: non-zero coefs = {'yoy_condition_change': np.float64(-0.0026695518745840506)}
  2008: non-zero coefs = {'yoy_condition_change': np.float64(-0.0009895524239146988)}
  2016: non-zero coefs = {'crop_condition_normalized': np.float64(-0.0005212155011474922)}
  2017: non-zero coefs = {'crop_condition_normalized': np.float64(-0.0012019721248159839)}
  2018: non-zero coefs = {'crop_condition_normalized': np.float64(-0.001338681266372675)}
  2019: non-zero coefs = {'crop_condition_normalized': np.float64(-0.0009241965190139918)}
  2022: non-zero coefs = {'crop_condition_normalized': np.float64(-0.00041384866799707007)}

OOS Results: IC=-0.1514, Hit=46.8%, N=293
  Sharpe=0.43, AnnRet=11.51%, MaxDD=-43.7

---
## 13. In-Sample vs Out-of-Sample Validation
Split data into IS (before 2023) and OOS (2023+) to check signal robustness.

In [23]:
OOS_START_YEAR = 2023

for cmd in COMMODITIES:
    m = master[cmd].copy()
    is_data = m[m["year"] < OOS_START_YEAR]
    oos_data = m[m["year"] >= OOS_START_YEAR]
    sig_cols = get_all_signal_cols(m) if "get_all_signal_cols" in dir() else KEY_SIGNALS
    sig_cols = [s for s in sig_cols if s in m.columns]
    print(f"\n{cmd} - IS: {len(is_data)} rows, OOS: {len(oos_data)} rows")
    results = []
    for sig in sig_cols:
        for label, data in [("IS", is_data), ("OOS", oos_data)]:
            valid = data[[sig, "fwd_5d_ret"]].dropna()
            if len(valid) < 10: continue
            ic = valid[sig].corr(valid["fwd_5d_ret"])
            hit = (np.sign(valid[sig]) == np.sign(valid["fwd_5d_ret"])).mean()
            results.append({"signal": sig, "period": label, "IC": ic, "hit_rate": hit, "n": len(valid)})
    rdf = pd.DataFrame(results)
    if len(rdf) == 0: continue
    pivot_ic = rdf.pivot(index="signal", columns="period", values="IC")
    pivot_hit = rdf.pivot(index="signal", columns="period", values="hit_rate")
    summary = pd.DataFrame({
        "IC_IS": pivot_ic.get("IS", pd.Series(dtype=float)),
        "IC_OOS": pivot_ic.get("OOS", pd.Series(dtype=float)),
        "IC_decay": pivot_ic.get("OOS", pd.Series(dtype=float)) - pivot_ic.get("IS", pd.Series(dtype=float)),
        "Hit_IS": pivot_hit.get("IS", pd.Series(dtype=float)),
        "Hit_OOS": pivot_hit.get("OOS", pd.Series(dtype=float)),
    }).sort_values("IC_IS", ascending=False, key=abs)
    print(f"  Top signals (IS vs OOS):")
    display(summary.head(15).style.format({"IC_IS":"{:.4f}","IC_OOS":"{:.4f}","IC_decay":"{:.4f}","Hit_IS":"{:.1%}","Hit_OOS":"{:.1%}"}))
    # PnL comparison for top signal
    top_sig = summary.index[0]
    if len(oos_data[[top_sig, "fwd_5d_ret"]].dropna()) > 5:
        result_oos, stats_oos = simulate_pnl_sum(oos_data, top_sig)
        print(f"  OOS PnL ({top_sig}): Sharpe={stats_oos['sharpe']:.2f}, AnnRet={stats_oos['ann_ret']:.2%}, MaxDD={stats_oos['max_dd']:.2%}")
        fig = go.Figure()
        fig.add_trace(go.Scatter(x=result_oos["signal_date"], y=result_oos["cum_strategy"], name="OOS Strategy"))
        fig.add_trace(go.Scatter(x=result_oos["signal_date"], y=result_oos["cum_buy_hold"], name="Buy&Hold", line=dict(dash="dash")))
        fig.update_layout(title=f"{cmd} OOS PnL ({top_sig})", template="plotly_white", height=350)
        fig.show()



CORN - IS: 502 rows, OOS: 60 rows
  Top signals (IS vs OOS):


,IC_IS,IC_OOS,IC_decay,Hit_IS,Hit_OOS
signal,,,,,
composite_rolling_ic,0.3716,0.4507,0.0791,63.6%,59.3%
composite_ic_weighted,0.1911,-0.0876,-0.2787,56.9%,60.8%
best_grid_composite,-0.1277,0.2751,0.4028,44.8%,54.4%
composite_equal,-0.1249,0.1052,0.2300,43.8%,50.0%
crop_condition_historical_surprise_z,-0.1172,0.1857,0.3028,46.1%,52.6%
crop_condition_historical_surprise,-0.1172,0.1857,0.3028,46.5%,52.6%
ge_zscore_harvest_ric,0.1142,0.0914,-0.0228,52.3%,53.3%
ge_zscore_growing,-0.1138,0.1227,0.2365,30.9%,31.0%
ge_zscore_growing_z,-0.1138,0.1227,0.2365,44.0%,44.8%


  OOS PnL (composite_rolling_ic): Sharpe=1.87, AnnRet=56.93%, MaxDD=-12.33%



SOYBEANS - IS: 448 rows, OOS: 56 rows
  Top signals (IS vs OOS):


,IC_IS,IC_OOS,IC_decay,Hit_IS,Hit_OOS
signal,,,,,
composite_rolling_ic,0.3762,0.4478,0.0716,62.6%,69.8%
ml_prediction,-0.1624,-0.0913,0.0711,45.8%,54.5%
composite_ic_weighted,0.1219,0.1380,0.0161,50.5%,38.7%
ge_zscore_growing_z,-0.0922,-0.1487,-0.0564,47.5%,40.0%
ge_zscore_growing,-0.0922,-0.1487,-0.0564,34.0%,30.9%
yoy_x_zscore_z,0.0920,-0.0429,-0.1349,53.3%,38.2%
yoy_x_zscore,0.0920,-0.0429,-0.1349,48.7%,32.4%
condition_momentum_3w_lag1,-0.0862,0.0199,0.1061,46.6%,54.5%
is_extreme_good_z,-0.0836,0.0769,0.1605,49.3%,56.4%


  OOS PnL (composite_rolling_ic): Sharpe=3.48, AnnRet=74.37%, MaxDD=-8.13%


---
## 14. Permutation Test + Multiple Testing Correction
Empirical significance via shuffled signal dates and Benjamini-Hochberg correction.

In [24]:
from scipy.stats import spearmanr

N_PERMUTATIONS = 500

for cmd in COMMODITIES:
    m = master[cmd].copy()
    sig_cols = get_all_signal_cols(m) if "get_all_signal_cols" in dir() else KEY_SIGNALS
    sig_cols = [s for s in sig_cols if s in m.columns]
    print(f"\n{cmd} - Permutation Test ({N_PERMUTATIONS} shuffles)")
    perm_results = []
    for sig in sig_cols:
        valid = m[[sig, "fwd_5d_ret"]].dropna()
        if len(valid) < 30: continue
        actual_ic = valid[sig].corr(valid["fwd_5d_ret"])
        # Permutation: shuffle signal values
        null_ics = []
        for _ in range(N_PERMUTATIONS):
            shuffled = valid[sig].sample(frac=1, replace=False).reset_index(drop=True)
            shuffled.index = valid.index
            null_ic = shuffled.corr(valid["fwd_5d_ret"])
            null_ics.append(null_ic)
        null_ics = np.array(null_ics)
        # p-value: fraction of null |IC| >= |actual IC|
        p_value = (np.abs(null_ics) >= np.abs(actual_ic)).mean()
        perm_results.append({"signal": sig, "actual_IC": actual_ic,
            "null_IC_mean": null_ics.mean(), "null_IC_std": null_ics.std(),
            "p_value": p_value, "n": len(valid)})
    if not perm_results: continue
    prdf = pd.DataFrame(perm_results).sort_values("p_value")
    # Benjamini-Hochberg correction
    m_tests = len(prdf)
    prdf = prdf.sort_values("p_value").reset_index(drop=True)
    prdf["bh_rank"] = range(1, len(prdf) + 1)
    prdf["bh_threshold"] = prdf["bh_rank"] / m_tests * 0.10  # 10% FDR
    prdf["significant_10pct_fdr"] = prdf["p_value"] <= prdf["bh_threshold"]
    print(f"\n  Tested {m_tests} signals. Significant at 10% FDR:")
    sig_df = prdf[prdf["significant_10pct_fdr"]]
    display(prdf.style.format({"actual_IC":"{:.4f}","null_IC_mean":"{:.4f}","null_IC_std":"{:.4f}","p_value":"{:.4f}","bh_threshold":"{:.4f}"}))
    if len(sig_df) > 0:
        print(f"  -> {len(sig_df)} signals survive BH correction at 10% FDR:")
        for _, r in sig_df.iterrows():
            print(f"     {r['signal']:45s} IC={r['actual_IC']:.4f}  p={r['p_value']:.4f}")
    else:
        print("  -> No signals survive BH correction. Consider larger sample or fewer tests.")
    # Histogram of null distribution for top signal
    top = prdf.iloc[0]
    top_null = []
    valid = m[[top["signal"], "fwd_5d_ret"]].dropna()
    for _ in range(N_PERMUTATIONS):
        shuffled = valid[top["signal"]].sample(frac=1, replace=False).reset_index(drop=True)
        shuffled.index = valid.index
        top_null.append(shuffled.corr(valid["fwd_5d_ret"]))
    fig = go.Figure()
    fig.add_trace(go.Histogram(x=top_null, name="Null IC", nbinsx=50))
    fig.add_vline(x=top["actual_IC"], line_dash="dash", line_color="red", annotation_text=f"Actual IC={top['actual_IC']:.4f}")
    fig.update_layout(title=f"{cmd} - Null IC Distribution ({top['signal']})", template="plotly_white", height=350)
    fig.show()



CORN - Permutation Test (500 shuffles)

  Tested 55 signals. Significant at 10% FDR:


,signal,actual_IC,null_IC_mean,null_IC_std,p_value,n,bh_rank,bh_threshold,significant_10pct_fdr
0,composite_rolling_ic,0.3779,-0.0027,0.0438,0.0000,472,1,0.0018,True
1,composite_ic_weighted,0.1644,0.0022,0.0458,0.0000,459,2,0.0036,True
2,condition_momentum_3w_lag1_z,-0.1076,-0.0006,0.0411,0.0080,536,3,0.0055,False
3,condition_momentum_3w_lag1,-0.1076,-0.0009,0.0406,0.0140,536,4,0.0073,False
4,ge_zscore_harvest_ric,0.1141,-0.0007,0.0462,0.0140,517,5,0.0091,False
5,condition_momentum_3w_lag2,-0.1018,-0.0011,0.0429,0.0160,535,6,0.0109,False
6,composite_equal,-0.1056,0.0002,0.0430,0.0180,562,7,0.0127,False
7,best_grid_composite,-0.0941,0.0009,0.0426,0.0300,535,8,0.0145,False
8,crop_condition_historical_surprise_z,-0.0885,-0.0009,0.0411,0.0380,532,9,0.0164,False
9,crop_condition_historical_surprise,-0.0885,0.0000,0.0428,0.0460,532,10,0.0182,False


  -> 2 signals survive BH correction at 10% FDR:
     composite_rolling_ic                          IC=0.3779  p=0.0000
     composite_ic_weighted                         IC=0.1644  p=0.0000



SOYBEANS - Permutation Test (500 shuffles)

  Tested 54 signals. Significant at 10% FDR:


,signal,actual_IC,null_IC_mean,null_IC_std,p_value,n,bh_rank,bh_threshold,significant_10pct_fdr
0,ge_zscore_planting,nan,nan,nan,0.0000,476,1,0.0019,True
1,composite_rolling_ic,0.3804,-0.0018,0.0488,0.0000,438,2,0.0037,True
2,ml_prediction,-0.1514,-0.0010,0.0563,0.0020,293,3,0.0056,True
3,composite_ic_weighted,0.1185,-0.0005,0.0514,0.0240,397,4,0.0074,False
4,ge_zscore_growing_z,-0.0976,0.0005,0.0470,0.0280,476,5,0.0093,False
5,ge_zscore_growing,-0.0976,-0.0009,0.0457,0.0320,476,6,0.0111,False
6,best_grid_composite,0.0905,-0.0040,0.0437,0.0340,492,7,0.0130,False
7,condition_momentum_3w_lag2,-0.0855,-0.0008,0.0474,0.0640,477,8,0.0148,False
8,condition_momentum_3w_lag2_z,-0.0855,-0.0002,0.0446,0.0640,477,9,0.0167,False
9,ge_zscore_z,-0.0791,-0.0024,0.0448,0.0640,476,10,0.0185,False


  -> 3 signals survive BH correction at 10% FDR:
     ge_zscore_planting                            IC=nan  p=0.0000
     composite_rolling_ic                          IC=0.3804  p=0.0000
     ml_prediction                                 IC=-0.1514  p=0.0020


##### Debug

In [25]:
a = master['CORN']
a[a['signal_date']>'2023-01-01']['fwd_5d_ret'].head()

502    0.017250
503    0.018946
504    0.140590
505   -0.146322
506   -0.120321
Name: fwd_5d_ret, dtype: float64

In [26]:
a.columns

Index(['signal_date', 'crop_condition_historical_surprise',
       'crop_condition_normalized', 'crop_condition_normalized_diff',
       'condition_momentum_3w', 'condition_acceleration',
       'yoy_condition_change', 'is_extreme_good', 'is_extreme_bad',
       'ge_zscore', 'ge_momentum_3w', 'good_excellent_pct', 'condition_index',
       'date', 'value', 'fwd_5d_ret', 'fwd_10d_ret', 'fwd_20d_ret', 'month',
       'year', 'season', 'date_vol', 'realized_vol_20d', 'vol_regime',
       'date_trend', 'price_20d_momentum', 'trend_regime',
       'surprise_x_momentum', 'zscore_x_accel', 'level_x_momentum',
       'yoy_x_zscore', 'crop_condition_historical_surprise_lag1',
       'crop_condition_historical_surprise_lag2', 'condition_momentum_3w_lag1',
       'condition_momentum_3w_lag2', 'ge_zscore_lag1', 'ge_zscore_lag2',
       'surprise_diff', 'vol_adj_diff', 'ge_zscore_growing',
       'ge_zscore_harvest', 'ge_zscore_planting', 'extreme_bad_deteriorating',
       'extreme_good_improving'

In [27]:
master['WINTER WHEAT'].tail()

KeyError: 'WINTER WHEAT'